In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
# Load training & validation data
train_data_path = "../Data/week5_wide_data.csv"
val_data_path = "../Data/val_data.csv"

train_df = pd.read_csv(train_data_path)
val_df = pd.read_csv(val_data_path)

# Convert val data to wide format
val_df_grouped = val_df.groupby(["country_name", "year", "series_name"], as_index=False)["value"].mean()
val_df_wide = val_df_grouped.pivot(index=["country_name", "year"], columns="series_name", values="value").reset_index()
val_df_wide.columns.name = None

# Define target variable
target_variable = "Life expectancy at birth, total (years)"

# Define features and exclude non-numeric column
feature_columns = [col for col in train_df.columns if col not in ["country_name", "year", target_variable]]
common_features = list(set(feature_columns) & set(val_df_wide.columns))

# Prepare data
X_train, y_train = train_df[common_features], train_df[target_variable]
X_val, y_val = val_df_wide[common_features], val_df_wide[target_variable]

# Fill missing values
X_train.fillna(X_train.mean(), inplace=True)
X_val.fillna(X_val.mean(), inplace=True)


/var/folders/z6/vcd1nm395kx4c8bz82wk7hdw0000gn/T/ipykernel_10086/4262633202.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.fillna(X_train.mean(), inplace=True)
/var/folders/z6/vcd1nm395kx4c8bz82wk7hdw0000gn/T/ipykernel_10086/4262633202.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_val.fillna(X_val.mean(), inplace=True)


In [3]:
# Define function to train and evaluate XGBoost models
def evaluate_xgboost(model, X_train, y_train, X_val, y_val, model_name):
    model.fit(X_train, y_train)
    train_preds = model.predict(X_train)
    val_preds = model.predict(X_val)

    metrics = {
        "Model": model_name,
        "Train RMSE": np.sqrt(mean_squared_error(y_train, train_preds)),
        "Validation RMSE": np.sqrt(mean_squared_error(y_val, val_preds)),
        "Train R²": r2_score(y_train, train_preds),
        "Validation R²": r2_score(y_val, val_preds),
    }

    print(f"{model_name} - Train RMSE: {metrics['Train RMSE']:.4f}, Validation RMSE: {metrics['Validation RMSE']:.4f}")
    return metrics


In [4]:
# Variation 1: default XGBoost with early stopping
xgb1 = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, 
                    colsample_bytree=1.0, subsample=1.0)  # No feature subsampling
metrics1 = evaluate_xgboost(xgb1, X_train, y_train, X_val, y_val, "XGB default with early stopping")

# Variation 2: XGBoost with feature subsampling
xgb2 = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42,
                    colsample_bytree=0.8, subsample=0.8)  # Feature and data subsampling added
metrics2 = evaluate_xgboost(xgb2, X_train, y_train, X_val, y_val, "XGB feature subsampling")

# Variation 3: XGBoost with 300 trees, low regularization & learning rate
xgb3 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.005,
    max_depth=2,
    colsample_bytree=0.6,
    subsample=0.6,
    reg_alpha=2.0,
    reg_lambda=2.0,
    random_state=42
)

metrics3 = evaluate_xgboost(xgb3, X_train, y_train, X_val, y_val, "XGB regularized")
metrics_df = pd.DataFrame([metrics1, metrics2, metrics3])

# Results
metrics_df

XGB default with early stopping - Train RMSE: 0.0695, Validation RMSE: 0.0840
XGB feature subsampling - Train RMSE: 0.4378, Validation RMSE: 0.3015
XGB regularized - Train RMSE: 3.0677, Validation RMSE: 2.5357


,Model,Train RMSE,Validation RMSE,Train R²,Validation R²
0,XGB default with early stopping,0.069544,0.083957,0.999944,0.999895
1,XGB feature subsampling,0.437757,0.301470,0.997800,0.998648
2,XGB regularized,3.067678,2.535737,0.891966,0.904340


In [ ]:
# Load test data
test_data_path = "../Data/test_data.csv"
test_df = pd.read_csv(test_data_path)

# Convert it to wide format
test_df_grouped = test_df.groupby(["country_name", "year", "series_name"], as_index=False)["value"].mean()
test_df_wide = test_df_grouped.pivot(index=["country_name", "year"], columns="series_name", values="value").reset_index()
test_df_wide.columns.name = None  # Remove column index name

# Align with training data
X_test = test_df_wide[X_train.columns]
y_test = test_df_wide[target_variable]

# Evaluate XGBoost regularized
metrics_test = evaluate_xgboost(xgb3, X_train, y_train, X_test, y_test, "Final Model - Test Set")

# Test results
print("\n Final Model performance on test data:")
print(f"Train RMSE: {metrics_test['Train RMSE']:.4f}, Validation RMSE: {metrics_test['Validation RMSE']:.4f}")
print(f"Train R²: {metrics_test['Train R²']:.4f}, Validation R²: {metrics_test['Validation R²']:.4f}")

test_results_df = pd.DataFrame([metrics_test])
print("\n Test set results:")
print(test_results_df.to_string(index=False))


Final Model - Test Set - Train RMSE: 3.0677, Validation RMSE: 2.6919

 Final Model performance on test data:
Train RMSE: 3.0677, Validation RMSE: 2.6919
Train R²: 0.8920, Validation R²: 0.8916

 Test set results:
                 Model  Train RMSE  Validation RMSE  Train R²  Validation R²
Final Model - Test Set    3.067678         2.691949  0.891966       0.891576
